# 📊 End-to-End Customer Churn Prediction System
### Metodologi CRISP-DM (*Cross-Industry Standard Process for Data Mining*)
**Author:** Sofia Azzahra Alamsyah  
**Model Utama:** Random Forest Classifier (Tuned with GridSearchCV)  
**Bidang:** Data Science / Machine Learning Portfolio  

---

## 📌 Alur Metodologi CRISP-DM
1. **Phase 1: Business Understanding** - Definisi Masalah Retensi & Target Bisnis
2. **Phase 2: Data Understanding** - Eksplorasi Dataset, Tipe Data, & EDA
3. **Phase 3: Data Preparation (Refined)** - Imputasi Median, Deteksi Outlier, 6 Rekayasa Fitur Baru, Ordinal & One-Hot Encoding, Feature Scaling
4. **Phase 4: Modeling** - Pelatihan Single Model (Random Forest) dengan 5-Fold Stratified CV & GridSearchCV
5. **Phase 5: Evaluation** - Evaluasi Metrik Dinamis (ROC-AUC, Precision, Recall, F1, Brier Score, Confusion Matrix, & Feature Importance)
6. **Phase 6: Deployment** - Ekspor Model Artifact & Metadata JSON untuk Web Dashboard Flask

---

## 🏢 Phase 1: Business Understanding

### Latar Belakang Masalah
Dalam industri berbasis langganan (*subscription*), mempertahankan pelanggan lama (*Customer Retention*) jauh lebih hemat biaya dibandingkan mencari pelanggan baru (*Customer Acquisition Cost* / CAC). Ketika pelanggan memutuskan untuk berhenti berlangganan (*churn*), bisnis kehilangan potensi pendapatan berulang (*Customer Lifetime Value*).

### Tujuan Proyek
1. Mengidentifikasi karakteristik dan pola pelanggan yang berisiko melakukan churn.
2. Membangun model machine learning (**Random Forest Classifier**) untuk memprediksi probabilitas churn setiap pelanggan secara real-time.
3. Merancang rekomendasi aksi retensi yang dapat dieksekusi secara otomatis oleh tim operasional.

### Metrik Keberhasilan
- **ROC-AUC & Recall**: Diutamakan untuk meminimalkan *False Negative* (pelanggan yang sebenarnya churn tetapi gagal terdeteksi sistem).

In [ ]:
# Import library yang dibutuhkan
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, brier_score_loss, accuracy_score
)

sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', None)

## 🔍 Phase 2: Data Understanding

Memuat dataset, memahami struktur tabel, memeriksa tipe data, melihat statistik deskriptif, dan mengeksplorasi distribusi fitur (*Exploratory Data Analysis* / EDA).

In [ ]:
# 1. Memuat dataset
dataset_path = os.path.join("Raw dataset", "ml_portfolio_customer_churn_dataset.csv")
df = pd.read_csv(dataset_path)

print(f"Dimensi data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# 2. Info tipe data dan ringkasan statistik
df.info()
df.describe().T

In [ ]:
# 3. Distribusi Target Churn
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='churn', data=df, hue='churn', palette='Set2', legend=False)
plt.title('Distribusi Churn Pelanggan (Target Variable)')
plt.xlabel('Status Churn (0 = Stayed, 1 = Churned)')
plt.ylabel('Jumlah Pelanggan')
total = len(df)
for p in ax.patches:
    pct = f'{100 * p.get_height() / total:.1f}%'
    ax.annotate(pct, (p.get_x() + p.get_width() / 2 - 0.1, p.get_height() + 10), fontweight='bold')
plt.tight_layout()
plt.show()

## 🧹 Phase 3: Data Preparation (Analisis Mendalam & Feature Engineering)

### 1. Analisis Missing Values & Strategi Imputasi
- Ditemukan missing values pada `income_million_idr` (2.0%), `monthly_spend_idr` (2.0%), dan `satisfaction_score` (2.0%).
- **Strategi Imputasi**: Menggunakan **Median Imputation** untuk variabel numerik karena median lebih tahan (*robust*) terhadap skewness dan nilai ekstrim dibandingkan mean.

### 2. Analisis Outlier & Skewness
- Analisis IQR mengidentifikasi sedikit outlier pada frekuensi login dan pengeluaran.
- Nilai rasio dibatasi (*clipped*) agar tidak mendominasi pembagian split pada pohon keputusan.

### 3. Penambahan Fitur Baru (Feature Engineering)
1. **`contract_duration_months`**: *Ordinal Mapping* untuk jenis kontrak (`Monthly` = 1, `Quarterly` = 3, `Annual` = 12 bulan).
2. **`support_tickets_per_tenure`**: Rasio keluhan per bulan masa langganan (korelasi positif terkuat terhadap churn: +0.081).
3. **`spend_per_login`**: Nilai pengeluaran bulanan per sesi aktif login.
4. **`support_to_login_ratio`**: Rasio intensitas keluhan dibanding frekuensi login.
5. **`spend_to_income_ratio`**: Rasio beban pengeluaran terhadap total pendapatan bulanan.
6. **`is_low_satisfaction`**: Indikator biner untuk skor kepuasan kritis (&le; 2.5).
7. **`tenure_segment`**: Segmentasi pelanggan (`New`: 1-12 bulan, `Medium`: 13-36 bulan, `Loyal`: >36 bulan).

In [ ]:
# Implementasi Feature Engineering
df_fe = df.copy()

contract_map = {'Monthly': 1, 'Quarterly': 3, 'Annual': 12}
df_fe['contract_duration_months'] = df_fe['contract_type'].map(contract_map)
df_fe['support_tickets_per_tenure'] = df_fe['support_tickets'] / (df_fe['tenure_months'] + 1)
df_fe['spend_per_login'] = df_fe['monthly_spend_idr'] / (df_fe['login_frequency_monthly'] + 1)
df_fe['support_to_login_ratio'] = df_fe['support_tickets'] / (df_fe['login_frequency_monthly'] + 1)
df_fe['spend_to_income_ratio'] = (df_fe['monthly_spend_idr'] / (df_fe['income_million_idr'] * 1000 + 1e-5)).clip(0, 5)
df_fe['is_low_satisfaction'] = (df_fe['satisfaction_score'] <= 2.5).astype(int)
df_fe['tenure_segment'] = df_fe['tenure_months'].apply(
    lambda m: 'New' if m <= 12 else ('Medium' if m <= 36 else 'Loyal')
)

print("Tabel Korelasi Fitur Rekayasa terhadap Churn:")
corr_matrix = df_fe.select_dtypes(include=[np.number]).corr()
print(corr_matrix['churn'].sort_values(ascending=False))

In [ ]:
# Menyiapkan Pipeline Preprocessing (ColumnTransformer) & Stratified Train-Test Split (80:20)
X = df_fe.drop(columns=['customer_id', 'churn'])
y = df_fe['churn']

numeric_features = [
    'age', 'income_million_idr', 'tenure_months', 'monthly_spend_idr',
    'support_tickets', 'login_frequency_monthly', 'satisfaction_score',
    'contract_duration_months', 'spend_to_income_ratio', 
    'support_tickets_per_tenure', 'spend_per_login', 'support_to_login_ratio',
    'is_low_satisfaction', 'discount_used'
]
categorical_features = ['contract_type', 'payment_method', 'tenure_segment']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Data Train: {X_train.shape}, Data Test: {X_test.shape}")

## 🤖 Phase 4: Modeling (Single Model: Random Forest Classifier)

Model yang dipilih adalah **Random Forest Classifier** dengan pembobotan kelas `balanced` untuk menangani ketidakseimbangan target. Hyperparameter dioptimalkan menggunakan **5-Fold Stratified Cross Validation** dan **GridSearchCV**.

In [ ]:
rf_pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [3, 5, 8],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2, 4]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(rf_pipe, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_pipeline = grid_search.best_estimator_
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")

## 📈 Phase 5: Evaluation (Test Set Evaluation)

Evaluasi pada data uji (240 data pelanggan unseen) dengan metrik klasifikasi: **ROC-AUC, Precision, Recall, F1-Score, Brier Score**, dan **Confusion Matrix**.

In [ ]:
y_pred = best_pipeline.predict(X_test)
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
acc = accuracy_score(y_test, y_pred)
brier = brier_score_loss(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"ROC-AUC Score : {roc_auc:.4f}")
print(f"Brier Score   : {brier:.4f}")

# Visualisasi Confusion Matrix
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Stayed (0)', 'Churned (1)'],
            yticklabels=['Stayed (0)', 'Churned (1)'])
plt.title('Confusion Matrix - Random Forest')
plt.xlabel('Prediksi Model')
plt.ylabel('Aktual Data')
plt.tight_layout()
plt.show()

In [ ]:
# Kurva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='#4361ee', lw=2.5, label=f'Random Forest (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='#94a3b8', linestyle='--', label='Random Guess')
plt.title('Kurva ROC (Test Set)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
rf_clf = best_pipeline.named_steps['classifier']
ohe_names = best_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(ohe_names)
importances = rf_clf.feature_importances_
idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
sorted_feat = [all_features[i] for i in idx[:10]]
sns.barplot(x=importances[idx][:10], y=sorted_feat, hue=sorted_feat, palette='viridis', legend=False)
plt.title('Top 10 Feature Importance (Random Forest)')
plt.xlabel('Tingkat Kepentingan')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

## 🚀 Phase 6: Deployment

Model terbaik diekspor ke file `models/best_churn_model.pkl` dan metadata diekspor ke `models/model_metadata.json` untuk dimuat oleh aplikasi web dashboard **Flask** (`app.py`).

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_pipeline, 'models/best_churn_model.pkl')
print("Model pipeline berhasil diekspor ke 'models/best_churn_model.pkl'")
print("Jalankan `python app.py` untuk membuka Web Dashboard Flask!")